                                           ================================================================================
                                                          ARABIC DIALECT RECOGNITION SYSTEM
                                      ================================================================================
                                              Application: Deep Learning for Arabic Dialect Classification Version 1.0
                                    Developed by Research Team : Abdulrahman Al-Kannad & Moeen Al-Mahalify  Eide Al-Mahlaify 
                                            ================================================================================

# Step 0: Import Core Libraries(GPU)

In [ ]:
import os
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split

# إعدادات الصوت والميزات
SR = 22050
FILE_DURATION = 30.0
SEGMENTS = 5
SEGMENT_DURATION = FILE_DURATION / SEGMENTS
IMG_SIZE = (224, 224)

def extract_segments(y_audio, sr):
    segment_len = int(SEGMENT_DURATION * sr)
    segments = []
    
    for i in range(SEGMENTS):
        start = i * segment_len
        end = start + segment_len
        if end > len(y_audio):
            break
        segments.append(y_audio[start:end])
    return segments

def process_and_split(data_path, genres, save_dir, feature_type='mfcc13', test_size=0.2):
    X, y = [], []
    
    # Determine number of MFCCs if using MFCC
    N_MFCC = 13
    if feature_type == 'mfcc13':
        N_MFCC = 13
    elif feature_type == 'mfcc20':
        N_MFCC = 20
    elif feature_type == 'mfcc40':
        N_MFCC = 40
    
    for idx, genre in enumerate(genres):
        print(f"Processing Genre: {genre}")
        genre_path = os.path.join(data_path, genre)

        for filename in os.listdir(genre_path):
            if filename.endswith(".wav"):
                file_path = os.path.join(genre_path, filename)
                y_audio, _ = librosa.load(file_path, sr=SR, duration=FILE_DURATION)
                
                segments = extract_segments(y_audio, SR)
                
                for seg in segments:
                    if feature_type.startswith('mfcc'):
                        feature = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC)
                    elif feature_type == 'chroma':
                        feature = librosa.feature.chroma_stft(y=seg, sr=SR)
                    elif feature_type == 'mel_spectrogram':
                        mel = librosa.feature.melspectrogram(y=seg, sr=SR)
                        feature = librosa.power_to_db(mel)
                    else:
                        raise ValueError("Unknown feature type")
                    
                    feature = tf.image.resize(feature[..., np.newaxis], IMG_SIZE).numpy()
                    feature = np.repeat(feature, 3, axis=-1)

                    X.append(feature)
                    y.append(idx)

    X = np.array(X)
    y = np.array(y)

    print(f"\nTotal Samples After Segmentation: {len(X)}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=42, stratify=y
    )

    # Create save directory
    train_dir = os.path.join(save_dir, "train", feature_type)
    test_dir = os.path.join(save_dir, "test", feature_type)
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    np.save(os.path.join(train_dir, f"{feature_type}_X_train.npy"), X_train)
    np.save(os.path.join(train_dir, f"{feature_type}_y_train.npy"), y_train)
    np.save(os.path.join(test_dir, f"{feature_type}_X_test.npy"), X_test)
    np.save(os.path.join(test_dir, f"{feature_type}_y_test.npy"), y_test)

    print(f"\n=== Final Data Information for Feature: {feature_type} ===")
    print("Training Data Shape:", X_train.shape)
    print("Training Labels Shape:", y_train.shape)
    print("Testing Data Shape:", X_test.shape)
    print("Testing Labels Shape:", y_test.shape)

# ==========================
# Main execution
# ==========================
if __name__ == "__main__":
    # Ask user to choose feature
    features = ['mfcc13', 'mfcc20', 'mfcc40', 'chroma', 'mel_spectrogram']
    print("Available Features:")
    for i, feat in enumerate(features, 1):
        print(f"  {i}. {feat}")
    
    choice = input("Select feature by number (1-5): ").strip()
    try:
        feature_type = features[int(choice)-1]
    except:
        feature_type = 'mfcc40'  # default
    
    data_path = "/root/YMED/"
    genres = ['adani', 'hadrami', 'lahgi', 'sanani', 'tihami']
    save_dir = "/root/autodl-tmp/split_data/"

    process_and_split(data_path, genres, save_dir, feature_type=feature_type, test_size=0.2)


Available Features:
  1. mfcc13
  2. mfcc20
  3. mfcc40
  4. chroma
  5. mel_spectrogram


In [1]:
def configure_gpu():
    """Configure GPU settings and check availability"""
    print("="*60)
    print("Step 2: GPU Configuration")
    print("="*60)

    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
    os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
    
    gpus = tf.config.list_physical_devices('GPU')
    
    if gpus:
        try:
            logical_gpus = tf.config.list_logical_devices('GPU')
            
            if len(logical_gpus) != len(gpus):
                print("⚠ Virtual GPUs detected — skipping memory_growth setting.")
            else:
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)

            print(f"✓ {len(gpus)} GPU(s) detected")
            for i, gpu in enumerate(gpus):
                print(f"  GPU {i}: {gpu.name}")

        except Exception as e:
            print(f"✗ GPU configuration error: {e}")

    else:
        print("✗ No GPU found, using CPU")

    print(f"TensorFlow Version: {tf.__version__}")
    print("="*60)

    return len(gpus) > 0


# Step 1: Import Core Libraries(CPU)

# Step 2: # ========== IMPORT LIBRARIES ==========

In [2]:
# ========== IMPORT OTHER LIBRARIES ==========
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
import os
from tensorflow.keras.applications import ResNet50, VGG16, MobileNet
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import time
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNet, MobileNetV2, Xception
from tensorflow.keras.applications.mobilenet import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenetv2_preprocess
from tensorflow.keras.applications.xception import preprocess_input as xception_preprocess

from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Step 3: Define CNN Models

In [3]:
# ============================================================================ 
# MODEL DEFINITIONS - USING ONLY tf.keras (NO layers import needed)
# ============================================================================

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, Dense, Dropout, GlobalAveragePooling2D, Flatten, Conv2D, 
    MaxPooling2D, BatchNormalization, Activation, LayerNormalization, 
    MultiHeadAttention, Add, Reshape, GlobalAveragePooling1D, Embedding
)
from tensorflow.keras.applications import MobileNet, ResNet50, VGG16, EfficientNetB0
from tensorflow.keras.applications.mobilenet import preprocess_input
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout
from tensorflow.keras.layers import GlobalAveragePooling2D

def create_panns_cnn14_model(input_shape, num_classes):
    """
    PANNs CNN14 Original Architecture
    Exactly as described in the paper "PANNs: Large-Scale Pretrained Audio Neural Networks"
    
    Args:
        input_shape: (mel_bands, time_steps, channels) e.g., (64, 1000, 1)
        num_classes: number of output classes (527 for AudioSet)
    
    Returns:
        Keras model matching original PANNs CNN14
    """
    model = Sequential([
        # ==============================================
        # Block 1
        # ==============================================
        Conv2D(64, (3, 3), padding='same', activation='relu', input_shape=input_shape),
        Conv2D(64, (3, 3), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        Dropout(0.2),
        
        # ==============================================
        # Block 2
        # ==============================================
        Conv2D(128, (3, 3), padding='same', activation='relu'),
        Conv2D(128, (3, 3), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        Dropout(0.2),
        
        # ==============================================
        # Block 3
        # ==============================================
        Conv2D(256, (3, 3), padding='same', activation='relu'),
        Conv2D(256, (3, 3), padding='same', activation='relu'),
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        Dropout(0.2),
        
        # ==============================================
        # Block 4 (with dilation)
        # ==============================================
        Conv2D(512, (3, 3), padding='same', activation='relu', dilation_rate=(1, 1)),
        Conv2D(512, (3, 3), padding='same', activation='relu', dilation_rate=(2, 2)),
        MaxPooling2D(pool_size=(2, 2), strides=(2, 2)),
        Dropout(0.2),
        
        # ==============================================
        # Block 5 (with dilation)
        # ==============================================
        Conv2D(1024, (3, 3), padding='same', activation='relu', dilation_rate=(1, 1)),
        Conv2D(1024, (3, 3), padding='same', activation='relu', dilation_rate=(4, 4)),
        
        # ==============================================
        # Global Pooling
        # ==============================================
        GlobalAveragePooling2D(),
        
        # ==============================================
        # Fully Connected Layers (Classifier)
        # ==============================================
        Dense(4096, activation='relu'),
        Dropout(0.5),
        Dense(4096, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='sigmoid')  # sigmoid for multi-label classification
    ])
    
    return model
# =========================
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization
from tensorflow.keras.layers import MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.layers import Conv2D, Flatten, Reshape, Add
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization
from tensorflow.keras.layers import MultiHeadAttention
from tensorflow.keras.layers import Conv2D, Reshape, GlobalAveragePooling1D
import tensorflow as tf

 
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization
from tensorflow.keras.layers import MultiHeadAttention
from tensorflow.keras.layers import Conv2D, Reshape
import tensorflow as tf

# ============================================
# Custom Layer for Position Embedding (with full serialization support)
# ============================================
class PositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, num_patches, embedding_dim, **kwargs):
        super(PositionEmbedding, self).__init__(**kwargs)
        self.num_patches = num_patches
        self.embedding_dim = embedding_dim
        self.position_embedding = self.add_weight(
            name='position_embedding',
            shape=(1, num_patches, embedding_dim),
            initializer='random_normal',
            trainable=True
        )
    
    def call(self, inputs):
        return inputs + self.position_embedding
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'num_patches': self.num_patches,
            'embedding_dim': self.embedding_dim
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        return cls(**config)

# ============================================
# Custom Layer for Class Token (with full serialization support)
# ============================================
class ClassToken(tf.keras.layers.Layer):
    def __init__(self, embedding_dim, **kwargs):
        super(ClassToken, self).__init__(**kwargs)
        self.embedding_dim = embedding_dim
        self.class_token = self.add_weight(
            name='class_token',
            shape=(1, 1, embedding_dim),
            initializer='random_normal',
            trainable=True
        )
    
    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        class_token = tf.tile(self.class_token, [batch_size, 1, 1])
        return tf.concat([class_token, inputs], axis=1)
    
    def get_config(self):
        config = super().get_config()
        config.update({'embedding_dim': self.embedding_dim})
        return config
    
    @classmethod
    def from_config(cls, config):
        return cls(**config)

# ============================================
# Transformer Encoder Layer
# ============================================
class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, num_heads, mlp_dim, dropout_rate=0.1, **kwargs):
        super(TransformerEncoder, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.mlp_dim = mlp_dim
        self.dropout_rate = dropout_rate
        
        # Initialize layers in __init__ for proper serialization
        self.attention = MultiHeadAttention(num_heads=num_heads, key_dim=mlp_dim // num_heads)
        self.dropout1 = Dropout(dropout_rate)
        self.norm1 = LayerNormalization(epsilon=1e-6)
        
        self.dense1 = Dense(mlp_dim, activation='gelu')
        self.dropout2 = Dropout(dropout_rate)
        self.dense2 = Dense(mlp_dim)  # Output dimension matches input
        self.dropout3 = Dropout(dropout_rate)
        self.norm2 = LayerNormalization(epsilon=1e-6)
    
    def call(self, inputs):
        # Self-attention
        attn_output = self.attention(inputs, inputs)
        attn_output = self.dropout1(attn_output)
        x1 = self.norm1(inputs + attn_output)
        
        # MLP
        mlp_output = self.dense1(x1)
        mlp_output = self.dropout2(mlp_output)
        mlp_output = self.dense2(mlp_output)
        mlp_output = self.dropout3(mlp_output)
        
        # Output
        outputs = self.norm2(x1 + mlp_output)
        return outputs
    
    def get_config(self):
        config = super().get_config()
        config.update({
            'num_heads': self.num_heads,
            'mlp_dim': self.mlp_dim,
            'dropout_rate': self.dropout_rate
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        return cls(**config)

# ============================================================================
# Official Pretrained AST - Wrapper
# ============================================================================

def create_ast_base_model(input_shape=None, num_classes=5, device=None):
    """
    Official pretrained Audio Spectrogram Transformer (AST).

    Model:
        MIT/ast-finetuned-audioset-10-10-0.4593

    Important:
        - This is NOT a Keras model.
        - This is official pretrained AST from Hugging Face.
        - input_shape is ignored.
        - num_classes is kept only for compatibility with your model registry.
        - AST expects raw waveform, not MFCC/chroma/mel 224x224x3 images.
        - Sampling rate should be 16000 Hz.
        - Use it as an embedding extractor, then train SVM or Dense classifier.
    """

    import torch
    from transformers import ASTFeatureExtractor, ASTModel

    model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    feature_extractor = ASTFeatureExtractor.from_pretrained(model_name)
    ast_model = ASTModel.from_pretrained(model_name)

    ast_model.to(device)
    ast_model.eval()

    print("=" * 70)
    print("Official pretrained AST loaded successfully.")
    print("Model:", model_name)
    print("Device:", device)
    print("Note: This model is not trained using model.fit().")
    print("Use raw waveform -> AST embedding -> classifier.")
    print("=" * 70)

    return feature_extractor, ast_model, device


# ============================================================================
# Official AST Embedding Extractor
# ============================================================================

def extract_ast_embedding(audio_segment, feature_extractor, ast_model, device, sr=16000):
    """
    Extract AST CLS-token embedding from one raw audio segment.

    Input:
        audio_segment: raw waveform, shape = (audio_samples,)
        sr: sampling rate, default 16000 Hz

    Output:
        embedding: numpy array, shape = (768,)
    """

    import torch
    import numpy as np

    audio_segment = audio_segment.astype(np.float32)

    inputs = feature_extractor(
        audio_segment,
        sampling_rate=sr,
        return_tensors="pt"
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = ast_model(**inputs)

    embedding = outputs.last_hidden_state[:, 0, :]

    return embedding.detach().cpu().numpy()[0]




# ============================================================================
# Official Audio Pretrained Models Dictionary
# Use this separately on raw WAV audio, NOT inside mfcc13/mfcc20 loop
# ============================================================================

 
# =========================
# EfficientNet-B0 Model
# =========================
def create_efficientnet_b0_model(input_shape, num_classes):
    """
    EfficientNet-B0 transfer-learning baseline.

    Compatible with your current pipeline:
        input_shape = (224, 224, 3)
        labels = one-hot encoded
        loss = categorical_crossentropy

    Notes:
        - Uses ImageNet-pretrained EfficientNet-B0.
        - Suitable for MFCC13, MFCC20, MFCC40, Chroma, Mel-spectrogram,
          and FilterBank features after resizing to 224x224x3.
    """

    inputs = Input(shape=input_shape, name="input_feature_image")

    # Safe preprocessing for ImageNet-pretrained EfficientNet.
    x = efficientnet_preprocess(inputs)

    base_model = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=input_shape
    )

    # First stage: freeze backbone for stable transfer learning.
    base_model.trainable = False

    x = base_model(x, training=False)

    x = GlobalAveragePooling2D(name="global_average_pooling")(x)

    x = Dense(512, activation="relu", name="fc_512")(x)
    x = BatchNormalization(name="bn_512")(x)
    x = Dropout(0.5, name="dropout_512")(x)

    x = Dense(256, activation="relu", name="fc_256")(x)
    x = BatchNormalization(name="bn_256")(x)
    x = Dropout(0.4, name="dropout_256")(x)

    outputs = Dense(num_classes, activation="softmax", name="classification_output")(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="EfficientNetB0_Pretrained"
    )

    return model


# =========================
# AlexNet
# =========================
def create_alexnet_model(input_shape, num_classes):
    model = Sequential([
        Conv2D(64, (11,11), strides=(4,4), activation='relu', input_shape=input_shape),
        MaxPooling2D((3,3), strides=(2,2)),
        Conv2D(192, (5,5), padding='same', activation='relu'),
        MaxPooling2D((3,3), strides=(2,2)),
        Conv2D(384, (3,3), padding='same', activation='relu'),
        Conv2D(256, (3,3), padding='same', activation='relu'),
        Conv2D(256, (3,3), padding='same', activation='relu'),
        MaxPooling2D((3,3), strides=(2,2)),
        Flatten(),
        Dense(1024, activation='relu'),
        Dropout(0.5),
        Dense(1024, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model



# =========================
# AlexNet-style CNN (custom-moeen) - Best Stable Version
# No new layers added
# =========================
def create_alexnet_style_cnn(input_shape, num_classes):
    model = Sequential([
        Conv2D(
            64, (7, 7),
            strides=(2, 2),
            activation='relu',
            padding='same',
            kernel_initializer='he_normal',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4),
            input_shape=input_shape
        ),
        BatchNormalization(),
        MaxPooling2D((3, 3), strides=(2, 2)),
        Dropout(0.20),

        Conv2D(
            192, (5, 5),
            activation='relu',
            padding='same',
            kernel_initializer='he_normal',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        BatchNormalization(),
        MaxPooling2D((3, 3), strides=(2, 2)),
        Dropout(0.20),

        Conv2D(
            384, (3, 3),
            activation='relu',
            padding='same',
            kernel_initializer='he_normal',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        BatchNormalization(),

        Conv2D(
            256, (3, 3),
            activation='relu',
            padding='same',
            kernel_initializer='he_normal',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        BatchNormalization(),

        Conv2D(
            256, (3, 3),
            activation='relu',
            padding='same',
            kernel_initializer='he_normal',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        BatchNormalization(),
        MaxPooling2D((3, 3), strides=(2, 2)),
        Dropout(0.25),

        Flatten(),

        Dense(
            512,
            activation='relu',
            kernel_initializer='he_normal',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        Dropout(0.40),

        Dense(
            256,
            activation='relu',
            kernel_initializer='he_normal',
            kernel_regularizer=tf.keras.regularizers.l2(1e-4)
        ),
        Dropout(0.35),

        Dense(num_classes, activation='softmax')
    ])

    return model

# =========================
# Dictionary of Models
# =========================
MODELS = {
    'create_panns_cnn14': create_panns_cnn14_model,
    'create_efficientnet_b0': create_efficientnet_b0_model,
    'alexnet': create_alexnet_model,
    'custom_cnn': create_alexnet_style_cnn
}

# Official audio pretrained models must be used separately on raw WAV audio.
# They are intentionally NOT included in MODELS because the training loop uses model.fit().
OFFICIAL_AUDIO_MODELS = {
    'official_ast': create_ast_base_model
}

# Step 4: Data Augmentation Functions

In [4]:

def augment_data(X, y, augmentation_factor=4):
    """Augment data using TensorFlow's ImageDataGenerator"""
    print(f"Augmenting data by factor {augmentation_factor}...")
    print(f"Original shape: {X.shape}")
    
    datagen = ImageDataGenerator(
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    
    original_samples = len(X)
    total_samples_needed = original_samples * augmentation_factor
    additional_samples_needed = total_samples_needed - original_samples
    
    X_augmented = np.copy(X)
    y_augmented = np.copy(y)
    
    batch_size = min(16, original_samples)
    batches_needed = int(np.ceil(additional_samples_needed / batch_size))
    
    for i in range(batches_needed):
        idx = np.random.randint(0, original_samples, batch_size)
        X_batch = X[idx]
        y_batch = y[idx]
        
        X_aug_batch = np.array([datagen.random_transform(img) for img in X_batch])
        
        X_augmented = np.vstack([X_augmented, X_aug_batch])
        y_augmented = np.concatenate([y_augmented, y_batch])
        
        if len(X_augmented) >= total_samples_needed:
            X_augmented = X_augmented[:total_samples_needed]
            y_augmented = y_augmented[:total_samples_needed]
            break
    
    print(f"Augmented shape: {X_augmented.shape}")
    return X_augmented, y_augmented

def load_or_create_augmented_data(feature_name, split_data_dir, augmentation_factor=4):
    """Load existing augmented data or create new"""
    train_dir = os.path.join(split_data_dir, 'train', feature_name)
    test_dir = os.path.join(split_data_dir, 'test', feature_name)
    aug_dir = os.path.join(split_data_dir, 'augmented', feature_name)
    
    os.makedirs(aug_dir, exist_ok=True)
    
    aug_train_path = os.path.join(aug_dir, f'{feature_name}_X_train_augmented.npy')
    
    if os.path.exists(aug_train_path):
        print(f"Loading augmented data from: {aug_dir}")
        X_train = np.load(aug_train_path)
        y_train = np.load(os.path.join(aug_dir, f'{feature_name}_y_train_augmented.npy'))
        X_test = np.load(os.path.join(aug_dir, f'{feature_name}_X_test.npy'))
        y_test = np.load(os.path.join(aug_dir, f'{feature_name}_y_test.npy'))
        return X_train, X_test, y_train, y_test, True
    
    # Load original data
    X_train = np.load(os.path.join(train_dir, f'{feature_name}_X_train.npy'))
    y_train = np.load(os.path.join(train_dir, f'{feature_name}_y_train.npy'))
    X_test = np.load(os.path.join(test_dir, f'{feature_name}_X_test.npy'))
    y_test = np.load(os.path.join(test_dir, f'{feature_name}_y_test.npy'))
    
    # Augment data
    X_train_aug, y_train_aug = augment_data(X_train, y_train, augmentation_factor)
    
    # Save augmented data
    np.save(aug_train_path, X_train_aug)
    np.save(os.path.join(aug_dir, f'{feature_name}_y_train_augmented.npy'), y_train_aug)
    np.save(os.path.join(aug_dir, f'{feature_name}_X_test.npy'), X_test)
    np.save(os.path.join(aug_dir, f'{feature_name}_y_test.npy'), y_test)
    
    return X_train_aug, X_test, y_train_aug, y_test, True

# Step 5: Training and Evaluation Functions

In [5]:

def train_model(model, X_train, y_train_cat, X_test, y_test_cat, 
                batch_size, epochs, callbacks):
    """Train the model with progress tracking"""
    print(f"\nStarting training with:")
    print(f"  Batch size: {batch_size}")
    print(f"  Epochs: {epochs}")
    print(f"  Training samples: {len(X_train)}")
    print(f"  Validation samples: {len(X_test)}")
    print("\nModel Summary:")
    model.summary()
    start_time = time.time()
    
    history = model.fit(
        X_train,
        y_train_cat,
        batch_size=batch_size,
        epochs=epochs,
        validation_data=(X_test, y_test_cat),
        callbacks=callbacks,
        verbose=1
    )
    
    training_time = time.time() - start_time
    print(f"\nTraining completed in {training_time/60:.2f} minutes")
    
    return history

def evaluate_model(model, X_test, y_test_cat):
    """Evaluate model performance"""
    loss, accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
    print(f"\nModel Evaluation:")
    print(f"  Test Loss: {loss:.4f}")
    print(f"  Test Accuracy: {accuracy:.4f}")
    return loss, accuracy

def generate_training_plots(history, model_name, feature_name, output_dir):
    """Generate training history plots"""
    plt.figure(figsize=(14, 5))
    
    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    plt.title(f'{model_name} - Accuracy ({feature_name})')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim([0, 1])
    
    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
    plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    plt.title(f'{model_name} - Loss ({feature_name})')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plot_path = os.path.join(output_dir, f'{model_name}_{feature_name}_training.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    return plot_path

# Step 6: Results Saving Functions

In [6]:

def save_model_results(model, model_name, feature_name, accuracy, history, 
                       output_dir, y_test, y_pred, genres):
    """Save all model results and artifacts"""
    
    # Create directories
    model_dir = os.path.join(output_dir, 'models', model_name, feature_name)
    plot_dir = os.path.join(output_dir, 'plots', model_name, feature_name)
    report_dir = os.path.join(output_dir, 'reports', model_name, feature_name)
    
    for directory in [model_dir, plot_dir, report_dir]:
        os.makedirs(directory, exist_ok=True)
    
    # Save model
    model_path = os.path.join(model_dir, f'{model_name}_{feature_name}_final.h5')
    model.save(model_path)
    
    # Save training history
    history_path = os.path.join(report_dir, f'{model_name}_{feature_name}_history.csv')
    pd.DataFrame(history.history).to_csv(history_path, index=False)
    
    # Generate and save plots
    plot_path = generate_training_plots(history, model_name, feature_name, plot_dir)
    
    # Generate confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=genres, yticklabels=genres)
    plt.title(f'Confusion Matrix - {model_name} ({feature_name})')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    cm_path = os.path.join(plot_dir, f'{model_name}_{feature_name}_confusion.png')
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Generate classification report
    report = classification_report(y_test, y_pred, target_names=genres, output_dict=True)
    report_df = pd.DataFrame(report).transpose()
    report_path = os.path.join(report_dir, f'{model_name}_{feature_name}_report.csv')
    report_df.to_csv(report_path)
    
    # Save summary
    summary = {
        'model': model_name,
        'feature': feature_name,
        'accuracy': float(accuracy),
        'training_time': history.history.get('time', [0])[-1],
        'best_epoch': len(history.history['val_loss']),
        'best_val_loss': min(history.history['val_loss']),
        'best_val_accuracy': max(history.history['val_accuracy'])
    }
    
    summary_path = os.path.join(report_dir, f'{model_name}_{feature_name}_summary.json')
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=4)
    
    return {
        'model_path': model_path,
        'plot_path': plot_path,
        'cm_path': cm_path,
        'report_path': report_path,
        'summary_path': summary_path,
        'accuracy': accuracy
    }

# Step 7: Main Training Function

In [7]:

def train_and_evaluate_single_model(model_name, model_func, feature_name, genres, 
                                   num_classes, input_shape, split_data_dir, 
                                   output_dir, config):
    """Train and evaluate a single model on a single feature.

    Updated for custom-moeen/custom_cnn:
    - uses the improved AlexNet-style CNN definition above;
    - uses a safer learning rate for custom_cnn;
    - saves and reloads the best checkpoint based on validation accuracy;
    - evaluates and reports using the best checkpoint, not the final epoch.
    """
    
    print(f"\n{'='*60}")
    print(f"Training {model_name} on {feature_name}")
    print(f"{'='*60}")
    
    # Load data
    if config['use_augmentation']:
        X_train, X_test, y_train, y_test, _ = load_or_create_augmented_data(
            feature_name, split_data_dir, config['augmentation_factor']
        )
    else:
        train_dir = os.path.join(split_data_dir, 'train', feature_name)
        test_dir = os.path.join(split_data_dir, 'test', feature_name)
        
        X_train = np.load(os.path.join(train_dir, f'{feature_name}_X_train.npy'))
        y_train = np.load(os.path.join(train_dir, f'{feature_name}_y_train.npy'))
        X_test = np.load(os.path.join(test_dir, f'{feature_name}_X_test.npy'))
        y_test = np.load(os.path.join(test_dir, f'{feature_name}_y_test.npy'))
    
    # Convert to categorical
    y_train_cat = to_categorical(y_train, num_classes=num_classes)
    y_test_cat = to_categorical(y_test, num_classes=num_classes)
    
    # Print dataset info
    print(f"\nDataset Information:")
    print(f"  X_train shape: {X_train.shape}")
    print(f"  X_test shape: {X_test.shape}")
    print(f"  Classes: {num_classes}")
    
    # Create model
    model = model_func(input_shape, num_classes)
    
    # Compile model
    # The custom CNN was unstable with larger LR; use 5e-5 for this model.
    if model_name in ['custom_cnn', 'custom_moeen']:
        learning_rate = 5e-5
    else:
        learning_rate = config['learning_rate']

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"  Learning rate: {learning_rate}")
    
    # Callbacks
    checkpoint_path = os.path.join(output_dir, f'best_{model_name}_{feature_name}.h5')
    
    callbacks = [
        ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_accuracy',
            save_best_only=True,
            mode='max',
            verbose=1
        ),
        EarlyStopping(
            monitor='val_accuracy',
            patience=18 if model_name in ['custom_cnn', 'custom_moeen'] else config['early_stopping_patience'],
            restore_best_weights=True,
            mode='max',
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=4,
            min_lr=1e-7,
            verbose=1
        )
    ]
    
    # Train model
    history = train_model(
        model, X_train, y_train_cat, X_test, y_test_cat,
        batch_size=config['batch_size'],
        epochs=config['epochs'],
        callbacks=callbacks
    )
    
    # Load and evaluate the best checkpoint, not the final epoch
    if os.path.exists(checkpoint_path):
        best_model = tf.keras.models.load_model(checkpoint_path)
        print(f"\nLoaded best checkpoint: {checkpoint_path}")
    else:
        print("\nWarning: checkpoint file was not found. Evaluating the current model instead.")
        best_model = model
    
    # Evaluate model
    loss, accuracy = evaluate_model(best_model, X_test, y_test_cat)
    
    # Generate predictions using the best model
    y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)
    
    # Save results using the best model
    results = save_model_results(
        best_model, model_name, feature_name, accuracy, history,
        output_dir, y_test, y_pred, genres
    )
    
    # Clear memory
    tf.keras.backend.clear_session()
    
    return {
        'model_name': model_name,
        'feature_name': feature_name,
        'accuracy': accuracy,
        'loss': loss,
        'training_time': history.history.get('time', [0])[-1],
        'results': results,
        'best_checkpoint': checkpoint_path
    }


# Step 8: User Configuration

In [8]:

def get_user_configuration():
    """Get all configuration parameters from user"""
    
    print("\n" + "="*60)
    print("Step 8: Training Configuration")
    print("="*60)
    
    config = {}
    
    # 1. Select models
    print("\nAvailable Models:")
    for i, (name, func) in enumerate(MODELS.items(), 1):
        print(f"  {i}. {name}")
    
    model_choice = input("\nSelect models (comma-separated numbers or 'all'): ").strip()
    if model_choice.lower() == 'all':
        selected_models = list(MODELS.keys())
    else:
        indices = [int(x.strip()) - 1 for x in model_choice.split(',')]
        selected_models = [list(MODELS.keys())[i] for i in indices if i < len(MODELS)]
    
    config['models'] = selected_models
    
    # 2. Select features
    available_features = ['mfcc13', 'mfcc20', 'mfcc40', 'chroma', 'mel_spectrogram','filterbank']
    print("\nAvailable Features:")
    for i, feat in enumerate(available_features, 1):
        print(f"  {i}. {feat}")
    
    feature_choice = input("\nSelect features (comma-separated numbers or 'all'): ").strip()
    if feature_choice.lower() == 'all':
        selected_features = available_features
    else:
        indices = [int(x.strip()) - 1 for x in feature_choice.split(',')]
        selected_features = [available_features[i] for i in indices if i < len(available_features)]
    
    config['features'] = selected_features
    
    # 3. Training parameters
    print("\nTraining Parameters:")
    
    config['epochs'] = int(input("Number of epochs (default=50): ") or 50)
    config['batch_size'] = int(input("Batch size (default=16): ") or 16)
    config['learning_rate'] = float(input("Learning rate (default=0.0001): ") or 0.0001)
    
     # 4. Augmentation
    use_aug = input("\nUse data augmentation? (yes/no, default=no): ").strip().lower() or 'no'
    config['use_augmentation'] = use_aug == 'yes'
    
    if config['use_augmentation']:
        config['augmentation_factor'] = int(input("Augmentation factor (default=4): ") or 4)
    else:
        config['augmentation_factor'] = 1
    
    # 5. Early stopping
    config['early_stopping_patience'] = int(input("\nEarly stopping patience (default=10): ") or 10)
    
    # 6. Output directory
    config['output_dir'] = input("\nOutput directory name (default='training_results'): ") or 'training_results'
    
    print("\nConfiguration Summary:")
    print(f"  Models: {', '.join(config['models'])}")
    print(f"  Features: {', '.join(config['features'])}")
    print(f"  Epochs: {config['epochs']}")
    print(f"  Batch Size: {config['batch_size']}")
    print(f"  Learning Rate: {config['learning_rate']}")
    print(f"  Augmentation: {'No' if config['use_augmentation'] else 'Yes'}")
    if config['use_augmentation']:
        print(f"  Augmentation Factor: {config['augmentation_factor']}")
    print(f"  Early Stopping Patience: {config['early_stopping_patience']}")
    print(f"  Output Directory: {config['output_dir']}")
    
    confirm = input("\nConfirm configuration? (yes/no): ").strip().lower()
    if confirm != 'yes':
        print("Configuration cancelled. Exiting.")
        return None
    
    return config

# Step 9: Batch Training Function

In [9]:

def train_all_models(config, genres, num_classes, input_shape, data_path):
    """Train all selected models on all selected features"""
    
    print("\n" + "="*60)
    print("Step 9: Starting Batch Training")
    print("="*60)
    
    all_results = {}
    
    for model_name in config['models']:
        model_func = MODELS[model_name]
        model_results = {}
        
        print(f"\n{'='*50}")
        print(f"Processing Model: {model_name}")
        print(f"{'='*50}")
        
        for feature_name in config['features']:
            print(f"\nFeature: {feature_name}")
            
            try:
                result = train_and_evaluate_single_model(
                    model_name=model_name,
                    model_func=model_func,
                    feature_name=feature_name,
                    genres=genres,
                    num_classes=num_classes,
                    input_shape=input_shape,
                    split_data_dir=data_path,
                    output_dir=config['output_dir'],
                    config=config
                )
                
                model_results[feature_name] = result
                print(f"✓ {model_name} on {feature_name}: Accuracy = {result['accuracy']:.4f}")
                
            except Exception as e:
                print(f"✗ Error training {model_name} on {feature_name}: {str(e)}")
                model_results[feature_name] = {'error': str(e)}
        
        all_results[model_name] = model_results
    
    return all_results

# Step 10: Generate Final Reports

In [10]:

def generate_final_reports(all_results, config, genres):
    """Generate comprehensive final reports"""
    
    print("\n" + "="*60)
    print("Step 10: Generating Final Reports")
    print("="*60)
    
    # Create summary dataframe
    summary_data = []
    
    for model_name, model_results in all_results.items():
        for feature_name, result in model_results.items():
            if 'accuracy' in result:
                summary_data.append({
                    'Model': model_name,
                    'Feature': feature_name,
                    'Accuracy': result['accuracy'],
                    'Loss': result.get('loss', None),
                    'Training_Time': result.get('training_time', None)
                })
    
    summary_df = pd.DataFrame(summary_data)
    
    # Save summary to CSV
    summary_path = os.path.join(config['output_dir'], 'summary_results.csv')
    summary_df.to_csv(summary_path, index=False)
    print(f"Summary saved to: {summary_path}")
    
    # Generate comparison plots
    if not summary_df.empty:
        plt.figure(figsize=(14, 8))
        
        # Bar plot for accuracy comparison
        plt.subplot(2, 2, 1)
        pivot_df = summary_df.pivot(index='Feature', columns='Model', values='Accuracy')
        pivot_df.plot(kind='bar', ax=plt.gca())
        plt.title('Accuracy by Model and Feature')
        plt.xlabel('Feature')
        plt.ylabel('Accuracy')
        plt.legend(title='Model')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        # Heatmap for accuracy
        plt.subplot(2, 2, 2)
        heatmap_data = summary_df.pivot(index='Model', columns='Feature', values='Accuracy')
        sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', ax=plt.gca())
        plt.title('Accuracy Heatmap')
        
        # Training time comparison
        plt.subplot(2, 2, 3)
        time_df = summary_df.pivot(index='Feature', columns='Model', values='Training_Time')
        time_df.plot(kind='bar', ax=plt.gca())
        plt.title('Training Time by Model and Feature')
        plt.xlabel('Feature')
        plt.ylabel('Time (seconds)')
        plt.legend(title='Model')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        # Best model per feature
        plt.subplot(2, 2, 4)
        best_models = summary_df.loc[summary_df.groupby('Feature')['Accuracy'].idxmax()]
        plt.bar(best_models['Feature'], best_models['Accuracy'], color='skyblue')
        plt.title('Best Model per Feature')
        plt.xlabel('Feature')
        plt.ylabel('Best Accuracy')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        # Add accuracy values on bars
        for i, row in best_models.iterrows():
            plt.text(i, row['Accuracy'] + 0.01, 
                    f"{row['Model']}\n{row['Accuracy']:.3f}", 
                    ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        comparison_path = os.path.join(config['output_dir'], 'model_comparison.png')
        plt.savefig(comparison_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Comparison plot saved to: {comparison_path}")
    
    # Save all results to JSON
    json_path = os.path.join(config['output_dir'], 'all_results.json')
    # Convert numpy values to Python types for JSON serialization
    json_results = {}
    for model_name, model_results in all_results.items():
        json_results[model_name] = {}
        for feature_name, result in model_results.items():
            if isinstance(result, dict):
                json_results[model_name][feature_name] = {
                    k: float(v) if isinstance(v, (np.float32, np.float64)) else v
                    for k, v in result.items()
                }
    
    with open(json_path, 'w') as f:
        json.dump(json_results, f, indent=4)
    
    print(f"All results saved to: {json_path}")
    
    # Print final summary
    print("\n" + "="*60)
    print("FINAL TRAINING SUMMARY")
    print("="*60)
    
    if not summary_df.empty:
        print(f"\nTotal models trained: {len(summary_df)}")
        print(f"Best overall accuracy: {summary_df['Accuracy'].max():.4f}")
        print(f"Average accuracy: {summary_df['Accuracy'].mean():.4f}")
        
        # Best model for each feature
        print("\nBest Model per Feature:")
        for feature in summary_df['Feature'].unique():
            feature_df = summary_df[summary_df['Feature'] == feature]
            best_row = feature_df.loc[feature_df['Accuracy'].idxmax()]
            print(f"  {feature}: {best_row['Model']} - {best_row['Accuracy']:.4f}")
    
    return summary_df

# Step 11: Main Execution Function

In [11]:
def main():
    """Main execution function"""
    
    print("\n" + "="*80)
    print("DEEP LEARNING MODEL TRAINING PIPELINE")
    print("="*80)
    
    # Step 1: Configure GPU
    gpu_available = configure_gpu()
    
    if not gpu_available:
        print("\n⚠ Warning: Training on CPU will be slower.")
        proceed = input("Continue with CPU? (yes/no): ").strip().lower()
        if proceed != 'yes':
            print("Exiting...")
            return
    
    # Step 2: Get user configuration
    config = get_user_configuration()
    if config is None:
        return
    
    # Step 3: Define constants
    genres = ['adeni', 'hadrami', 'lahji', 'sanaai', 'tihami']
    num_classes = len(genres)
    input_shape = (224, 224, 3)
    data_path = '/root/autodl-tmp/split_data/'  # Update this path if needed
    
    # Step 4: Check data availability
    print("\n" + "="*60)
    print("Checking Data Availability")
    print("="*60)
    
    for feature in config['features']:
        train_path = os.path.join(data_path, 'train', feature, f'{feature}_X_train.npy')
        if os.path.exists(train_path):
            print(f"✓ {feature}: Data found")
        else:
            print(f"✗ {feature}: Data not found at {train_path}")
            print("Please check your data path and file structure.")
            return
    
    # Step 5: Create output directory
    os.makedirs(config['output_dir'], exist_ok=True)
    
    # Step 6: Save configuration
    config_path = os.path.join(config['output_dir'], 'training_config.json')
    with open(config_path, 'w') as f:
        # Convert functions to string names for JSON
        save_config = config.copy()
        save_config['models'] = [str(m) for m in save_config['models']]
        json.dump(save_config, f, indent=4)
    
    print(f"\nConfiguration saved to: {config_path}")
    
    # Step 7: Train all models
    all_results = train_all_models(config, genres, num_classes, input_shape, data_path)
    
    # Step 8: Generate reports
    summary_df = generate_final_reports(all_results, config, genres)
    
    # Step 9: Display directory structure
    print("\n" + "="*60)
    print("Output Directory Structure")
    print("="*60)
    
    def print_dir_tree(startpath, prefix=""):
        try:
            contents = sorted(os.listdir(startpath))
            for i, item in enumerate(contents):
                path = os.path.join(startpath, item)
                is_last = (i == len(contents) - 1)
                connector = "└── " if is_last else "├── "
                print(prefix + connector + item)
                if os.path.isdir(path):
                    extension = "    " if is_last else "│   "
                    print_dir_tree(path, prefix + extension)
        except:
            pass
        
    
    print_dir_tree(config['output_dir'])
    
    # Step 10: Completion message
    print("\n" + "="*80)
    print("TRAINING PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*80)
    print(f"\nAll results saved in: {os.path.abspath(config['output_dir'])}")
    print("\nFiles generated:")
    print("  • summary_results.csv - Summary of all experiments")
    print("  • model_comparison.png - Comparison plots")
    print("  • all_results.json - Complete results in JSON format")
    print("  • training_config.json - Training configuration")
    print("  • models/ - Trained model files")
    print("  • plots/ - Training and evaluation plots")
    print("  • reports/ - Detailed reports and metrics")
    
    if summary_df is not None and not summary_df.empty:
        best_overall = summary_df.loc[summary_df['Accuracy'].idxmax()]
        print(f"\n★ Best Overall Performance:")
        print(f"  Model: {best_overall['Model']}")
        print(f"  Feature: {best_overall['Feature']}")
        print(f"  Accuracy: {best_overall['Accuracy']:.4f}")

# Run the pipeline
if __name__ == "__main__":
    main()


DEEP LEARNING MODEL TRAINING PIPELINE
Step 2: GPU Configuration
✗ GPU configuration error: Physical devices cannot be modified after being initialized
TensorFlow Version: 2.9.0

Step 8: Training Configuration

Available Models:
  1. create_panns_cnn14
  2. create_efficientnet_b0
  3. alexnet
  4. custom_cnn



Select models (comma-separated numbers or 'all'):  4



Available Features:
  1. mfcc13
  2. mfcc20
  3. mfcc40
  4. chroma
  5. mel_spectrogram
  6. filterbank



Select features (comma-separated numbers or 'all'):  6



Training Parameters:


Number of epochs (default=50):  
Batch size (default=16):  
Learning rate (default=0.0001):  

Use data augmentation? (yes/no, default=no):  

Early stopping patience (default=10):  

Output directory name (default='training_results'):  



Configuration Summary:
  Models: custom_cnn
  Features: filterbank
  Epochs: 50
  Batch Size: 16
  Learning Rate: 0.0001
  Augmentation: Yes
  Early Stopping Patience: 10
  Output Directory: training_results



Confirm configuration? (yes/no):  yes



Checking Data Availability
✓ filterbank: Data found

Configuration saved to: training_results/training_config.json

Step 9: Starting Batch Training

Processing Model: custom_cnn

Feature: filterbank

Training custom_cnn on filterbank

Dataset Information:
  X_train shape: (23000, 224, 224, 3)
  X_test shape: (1150, 224, 224, 3)
  Classes: 5
  Learning rate: 5e-05

Starting training with:
  Batch size: 16
  Epochs: 50
  Training samples: 23000
  Validation samples: 1150

Model Summary:
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 112, 112, 64)      9472      
                                                                 
 batch_normalization (BatchN  (None, 112, 112, 64)     256       
 ormalization)                                                   
                                                                 
 max_pooling2d (MaxPooling2

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



1438/1438 [==============================] - ETA: 0s - loss: 0.2263 - accuracy: 0.9883
Epoch 35: val_accuracy did not improve from 0.96609

Epoch 35: ReduceLROnPlateau reducing learning rate to 3.12499992105586e-06.
1438/1438 [==============================] - 22s 15ms/step - loss: 0.2263 - accuracy: 0.9883 - val_loss: 0.6308 - val_accuracy: 0.9583 - lr: 6.2500e-06
Epoch 36/50
1438/1438 [==============================] - ETA: 0s - loss: 0.2229 - accuracy: 0.9887
Epoch 36: val_accuracy did not improve from 0.96609
1438/1438 [==============================] - 23s 16ms/step - loss: 0.2229 - accuracy: 0.9887 - val_loss: 0.6112 - val_accuracy: 0.9609 - lr: 3.1250e-06
Epoch 37/50
1437/1438 [============================>.] - ETA: 0s - loss: 0.2232 - accuracy: 0.9888
Epoch 37: val_accuracy did not improve from 0.96609
1438/1438 [==============================] - 22s 15ms/step - loss: 0.2232 - accuracy: 0.9888 - val_loss: 0.6228 - val_accuracy: 0.9635 - lr: 3.1250e-06
Epoch 38/50
1436/1438 [===